In [ ]:
# Launch the interactive inference GUI
gnb.showInference(constrained_bn, size="15")

In [ ]:
# 1. Define the evidence (the nodes you want to "click")
# Make sure these strings exactly match your discretized categories
my_evidence = {
    'MET_min': 'High', 
    'IOB': 'High'
}

# 2. Pass the evidence directly into the visualizer!
# pyagrum will calculate the math and draw the updated graph
gnb.showInference(constrained_bn, evs=my_evidence, size="15")

In [ ]:
# 1. Create an Inference Engine for your trained network
ie = gum.LazyPropagation(constrained_bn)

# 2. Define a hypothetical patient scenario (The Evidence)
# Make sure the string states perfectly match the names of your discretized bins!
evidence = {
    'MET': 'Vigorous',               
    'startExerciseGlucoseLevel': 'Hypo Risk', 
    'IOB': 'High'                 
}

ie.setEvidence(evidence)
ie.makeInference()

# 3. Query the Outcome Node
print("--- PREDICTING POST-EXERCISE GLUCOSE ---")
target_node = 'minGlucosePostExercise'
posterior = ie.posterior(target_node)
print(posterior)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pyagrum as gum

# 1. Initialize the Inference Engine with your trained network
ie = gum.LazyPropagation(constrained_bn)

# 2. Define our Evidence Nodes (The variables we want to "fix")
# We extract the exact labels from your network to populate the dropdowns
met_states = list(constrained_bn.variableFromName('MET').labels())
iob_states = list(constrained_bn.variableFromName('IOB').labels())
start_glucose_states = list(constrained_bn.variableFromName('startExerciseGlucoseLevel').labels())

# We add a 'Not Fixed' option so the network can calculate baseline probabilities
drop_met = widgets.Dropdown(options=['Not Fixed'] + met_states, description='MET (Intensity):', style={'description_width': 'initial'})
drop_iob = widgets.Dropdown(options=['Not Fixed'] + iob_states, description='IOB (Insulin):', style={'description_width': 'initial'})
drop_glucose = widgets.Dropdown(options=['Not Fixed'] + start_glucose_states, description='Start Glucose:', style={'description_width': 'initial'})

# 3. Define our Target Node (The outcome we want to watch)
target_node = 'minGlucosePostExercise'
target_states = list(constrained_bn.variableFromName(target_node).labels())

# Create an output area for our dynamically updating chart
out_plot = widgets.Output()

# 4. The Core Update Function
# This runs every single time you change a dropdown!
def update_dashboard(*args):
    with out_plot:
        clear_output(wait=True)
        
        # Reset the engine to baseline
        ie.eraseAllEvidence()
        
        # Inject the "Fixed" evidence from the dropdowns
        if drop_met.value != 'Not Fixed': 
            ie.addEvidence('MET', drop_met.value)
        if drop_iob.value != 'Not Fixed': 
            ie.addEvidence('IOB', drop_iob.value)
        if drop_glucose.value != 'Not Fixed': 
            ie.addEvidence('startExerciseGlucoseLevel', drop_glucose.value)
            
        # Calculate the new probabilities!
        ie.makeInference()
        posterior = ie.posterior(target_node)
        
        # Extract the exact math probabilities
        probs = [posterior[i] * 100 for i in range(len(target_states))] # Convert to percentages
        
        # Draw the Bar Chart
        plt.figure(figsize=(8, 4))
        bars = plt.bar(target_states, probs, color='#2ca02c', edgecolor='black')
        plt.ylim(0, 100)
        plt.ylabel('Probability (%)', fontsize=12)
        plt.title(f'Predicted Outcome: {target_node}', fontsize=14, fontweight='bold')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add the exact percentage numbers on top of the bars
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold')
            
        plt.tight_layout()
        plt.show()

# 5. Connect the dropdowns to the update function
drop_met.observe(update_dashboard, names='value')
drop_iob.observe(update_dashboard, names='value')
drop_glucose.observe(update_dashboard, names='value')

# 6. Display the Dashboard!
ui = widgets.VBox([
    widgets.HTML("<h3>T1D Exercise Inference Engine</h3>"),
    widgets.HBox([drop_met, drop_iob, drop_glucose]),
    out_plot
])

display(ui)

# Trigger the initial baseline plot
update_dashboard()